---
**Curso:** Tecnicas Estadisticas Avanzadas para Mineria de Datos - ESTP1030  
**Maestria:** Maestria en Estadistica Aplicada - ESPOL  
**Profesor:** Dr. Andres Abad Robalino

**Integrantes del Grupo:**
- VITERI RAMBAY IRWIN ALBERTO
- VICTORES SOLEDISPA LOURDES VANESSA
- GONZALEZ VILORIA GILBERTO ENRIQUE

---

# Tema 1: ¿Vale la pena agregar una tercera capa oculta?
### Comparacion de arquitecturas MLP para prediccion de abandono de clientes

## 0. Instalacion de dependencias

In [1]:
import sys
!{sys.executable} -m pip install weasyprint --quiet
print('Listo.')

Listo.


## 1. Configuracion del entorno

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, base64

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, f1_score, accuracy_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

# Ruta de datos Windows
DATA_PATH = r'G:\Evaluacion\data\Churn-Telco-Customer.csv'

# Carpeta para figuras
FIG_DIR = r'G:\Evaluacion\figuras_reporte'
os.makedirs(FIG_DIR, exist_ok=True)

print('Entorno listo.')
print('Figuras en:', FIG_DIR)

Entorno listo.
Figuras en: G:\Evaluacion\figuras_reporte


## 2. Carga y exploracion de los datos

El dataset **Telco Customer Churn** recoge informacion de 7 043 clientes. La variable objetivo `Churn` indica si el cliente cancelo el servicio ese mes.

In [4]:
df = pd.read_csv(DATA_PATH)
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head()

Filas: 7,043  |  Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Distribucion de la variable objetivo

Aproximadamente el **73% no abandona** y el **27% si abandona**. Por eso priorizamos el **F1-Score** sobre la Accuracy.

In [5]:
conteo = df['Churn'].value_counts()
prop   = df['Churn'].value_counts(normalize=True) * 100

for clase, n in conteo.items():
    print(f'  {clase}: {n:,} clientes  ({prop[clase]:.1f}%)')

fig, ax = plt.subplots(figsize=(5, 4))
conteo.plot(kind='bar', color=['#2196F3', '#F44336'], edgecolor='black', ax=ax)
ax.set_title('Distribucion de Churn', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('Numero de clientes')
ax.set_xticklabels(['No abandona', 'Abandona'], rotation=0)
for i, v in enumerate(conteo.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig01_distribucion_churn.png'), dpi=120)
plt.show()
print('Figura 1 guardada.')

  No: 5,174 clientes  (73.5%)
  Yes: 1,869 clientes  (26.5%)
Figura 1 guardada.


## 3. Preprocesamiento

### 3.1 Limpieza
`TotalCharges` tiene 11 celdas vacias. Se eliminan (menos del 0.2% del dataset).

### 3.2 Codificacion
`customerID` eliminado. `Churn` a binario. One-hot encoding en 15 variables categoricas. Dataset: 20 -> **30 features**.

### 3.3 Particion y Escalamiento
Division **80/20** estratificada. StandardScaler ajustado solo en train para evitar data leakage.

In [6]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Filas con TotalCharges vacio: {df["TotalCharges"].isnull().sum()}')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
print(f'Dimensiones: {df.shape}')

df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
y = df['Churn']
X = df.drop(columns=['Churn'])
cols_cat = X.select_dtypes(include='object').columns.tolist()
X = pd.get_dummies(X, columns=cols_cat, drop_first=True)
print(f'Features: {X.shape[1]}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print(f'Train: {X_train_scaled.shape[0]:,} | Test: {X_test_scaled.shape[0]:,}')
print(f'Churn train: {y_train.mean():.1%} | Churn test: {y_test.mean():.1%}')

Filas con TotalCharges vacio: 11
Dimensiones: (7032, 21)
Features: 30
Train: 5,625 | Test: 1,407
Churn train: 26.6% | Churn test: 26.6%


## 4. Modelo Base: arquitectura (64, 32)

Dos capas ocultas. Hiperparametros identicos al Tema 1 para comparacion justa.

In [7]:
mlp_base = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu', solver='adam',
    alpha=0.0001, learning_rate_init=0.001,
    max_iter=300, random_state=RANDOM_STATE
)
mlp_base.fit(X_train_scaled, y_train)
print(f'Convergencia: {mlp_base.n_iter_} iter | Perdida: {mlp_base.loss_:.4f}')

y_pred_base = mlp_base.predict(X_test_scaled)
y_prob_base = mlp_base.predict_proba(X_test_scaled)[:, 1]
f1_base  = f1_score(y_test, y_pred_base)
acc_base = accuracy_score(y_test, y_pred_base)
auc_base = roc_auc_score(y_test, y_prob_base)

print('Reporte Base (64, 32)')
print('=' * 50)
print(classification_report(y_test, y_pred_base, target_names=['No abandona', 'Abandona']))
print(f'AUC-ROC: {auc_base:.4f}')

Convergencia: 300 iter | Perdida: 0.1761
Reporte Base (64, 32)
              precision    recall  f1-score   support

 No abandona       0.82      0.86      0.84      1033
    Abandona       0.55      0.48      0.51       374

    accuracy                           0.76      1407
   macro avg       0.68      0.67      0.67      1407
weighted avg       0.75      0.76      0.75      1407

AUC-ROC: 0.7677


In [8]:
cm_base = confusion_matrix(y_test, y_pred_base)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No abandona', 'Abandona'],
            yticklabels=['No abandona', 'Abandona'], ax=ax)
ax.set_title('Matriz de Confusion - Base (64, 32)', fontsize=12)
ax.set_xlabel('Prediccion'); ax.set_ylabel('Valor Real')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig02_cm_base.png'), dpi=120)
plt.show()
print('Figura 2 guardada.')

Figura 2 guardada.


## 5. Modelo Tema 1: arquitectura (128, 64, 32)

Se anade una primera capa de 128 neuronas. Todo lo demas es identico al modelo base.

In [9]:
mlp_t1 = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu', solver='adam',
    alpha=0.0001, learning_rate_init=0.001,
    max_iter=300, random_state=RANDOM_STATE
)
mlp_t1.fit(X_train_scaled, y_train)
print(f'Convergencia: {mlp_t1.n_iter_} iter | Perdida: {mlp_t1.loss_:.4f}')

y_pred_t1 = mlp_t1.predict(X_test_scaled)
y_prob_t1 = mlp_t1.predict_proba(X_test_scaled)[:, 1]
f1_t1  = f1_score(y_test, y_pred_t1)
acc_t1 = accuracy_score(y_test, y_pred_t1)
auc_t1 = roc_auc_score(y_test, y_prob_t1)

print('Reporte Tema 1 (128, 64, 32)')
print('=' * 50)
print(classification_report(y_test, y_pred_t1, target_names=['No abandona', 'Abandona']))
print(f'AUC-ROC: {auc_t1:.4f}')

Convergencia: 181 iter | Perdida: 0.1044
Reporte Tema 1 (128, 64, 32)
              precision    recall  f1-score   support

 No abandona       0.84      0.80      0.82      1033
    Abandona       0.50      0.57      0.53       374

    accuracy                           0.74      1407
   macro avg       0.67      0.68      0.67      1407
weighted avg       0.75      0.74      0.74      1407

AUC-ROC: 0.7590


In [10]:
cm_t1 = confusion_matrix(y_test, y_pred_t1)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_t1, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No abandona', 'Abandona'],
            yticklabels=['No abandona', 'Abandona'], ax=ax)
ax.set_title('Matriz de Confusion - Tema 1 (128, 64, 32)', fontsize=12)
ax.set_xlabel('Prediccion'); ax.set_ylabel('Valor Real')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig03_cm_t1.png'), dpi=120)
plt.show()
print('Figura 3 guardada.')

Figura 3 guardada.


## 6. Comparacion directa

In [11]:
resumen = pd.DataFrame({
    'Metrica'           : ['Accuracy', 'F1-Score (Abandona)', 'AUC-ROC'],
    'Base (64,32)'      : [f'{acc_base:.4f}', f'{f1_base:.4f}', f'{auc_base:.4f}'],
    'Tema1 (128,64,32)' : [f'{acc_t1:.4f}',  f'{f1_t1:.4f}',  f'{auc_t1:.4f}'],
})
print(resumen.to_string(index=False))
print(f'Cambio F1: {(f1_t1 - f1_base) / f1_base * 100:+.2f}%')

            Metrica Base (64,32) Tema1 (128,64,32)
           Accuracy       0.7555            0.7356
F1-Score (Abandona)       0.5100            0.5338
            AUC-ROC       0.7677            0.7590
Cambio F1: +4.68%


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, titulo, cmap in zip(
    axes, [cm_base, cm_t1],
    ['Modelo Base (64, 32)', 'Tema 1 (128, 64, 32)'],
    ['Blues', 'Oranges']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['No abandona', 'Abandona'],
                yticklabels=['No abandona', 'Abandona'], ax=ax)
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('Prediccion'); ax.set_ylabel('Valor Real')
plt.suptitle('Comparacion de Matrices de Confusion', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig04_cm_comparacion.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Figura 4 guardada.')

Figura 4 guardada.


In [13]:
fpr_base, tpr_base, _ = roc_curve(y_test, y_prob_base)
fpr_t1,   tpr_t1,   _ = roc_curve(y_test, y_prob_t1)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_base, tpr_base, color='#2196F3', lw=2, label=f'Base (64, 32) - AUC = {auc_base:.3f}')
ax.plot(fpr_t1,   tpr_t1,   color='#FF9800', lw=2, label=f'Tema 1 (128, 64, 32) - AUC = {auc_t1:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
ax.set_title('Curvas ROC: Base vs Tema 1', fontsize=13)
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Recall (Verdaderos Positivos)', fontsize=11)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig05_roc.png'), dpi=120)
plt.show()
print('Figura 5 guardada.')

Figura 5 guardada.


In [14]:
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ['Base\n(64, 32)', 'Tema 1\n(128, 64, 32)'],
    [f1_base, f1_t1],
    color=['#2196F3', '#FF9800'], edgecolor='black', width=0.45
)
for bar, val in zip(bars, [f1_base, f1_t1]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(f1_base, f1_t1) + 0.08)
ax.set_title('F1-Score sobre clase Abandona', fontsize=12)
ax.set_ylabel('F1-Score')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig06_f1_comparacion.png'), dpi=120)
plt.show()
print('Figura 6 guardada.')

Figura 6 guardada.


## 7. Analisis e Interpretacion

### Lo que dicen los numeros
F1 sube de 0.5100 a 0.5338 (+4.68%). Recall de la clase Abandona sube de 0.48 a 0.57 (+9pp). Accuracy baja de 0.7555 a 0.7356 por mas falsos positivos, intercambio aceptable en el negocio. AUC-ROC baja de 0.7677 a 0.7590.

### Por que ayudo la capa adicional
Con 30 features mixtas, la capa de 128 neuronas aprende combinaciones que el modelo de dos capas no puede formar directamente. Las capas 64 y 32 refinan esas representaciones.

### Por que la mejora es modesta
El churn depende de pocas variables clave (contrato, antiguedad, cargos) ya capturadas por el modelo base. Ninguna arquitectura trata el desbalance 73/27%.

### Conclusion
Agregar la tercera capa **si mejora** la deteccion (Recall +9pp, F1 +4.68%) con costo aceptable en Accuracy. La arquitectura **(128, 64, 32) es la mejor opcion**. El siguiente paso debe ser tratar el desbalance con `class_weight='balanced'`.

---
## 8. Generacion del PDF

Ejecuta esta celda despues de correr todas las anteriores. El PDF se guarda en `G:\Evaluacion\reporte_tema1_MLP.pdf`.

In [15]:
import base64, os
from weasyprint import HTML as WP_HTML

def img_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

F = {
    'dist'   : os.path.join(FIG_DIR, 'fig01_distribucion_churn.png'),
    'cm_base': os.path.join(FIG_DIR, 'fig02_cm_base.png'),
    'cm_t1'  : os.path.join(FIG_DIR, 'fig03_cm_t1.png'),
    'cm_cmp' : os.path.join(FIG_DIR, 'fig04_cm_comparacion.png'),
    'roc'    : os.path.join(FIG_DIR, 'fig05_roc.png'),
    'f1'     : os.path.join(FIG_DIR, 'fig06_f1_comparacion.png'),
}

tabla_html = resumen.to_html(index=False, border=0, classes='tbl')

CSS = '''
@page { size: A4; margin: 2cm 2.5cm; }
body { font-family: Georgia, serif; font-size: 11pt; color: #1a1a2e; line-height: 1.6; }
.portada { text-align: center; padding: 60px 0 40px; border-bottom: 3px solid #1565C0; page-break-after: always; }
.portada h1 { font-size: 20pt; color: #1565C0; }
.portada h2 { font-size: 13pt; color: #424242; font-weight: normal; }
.meta { background: #E3F2FD; border-radius: 8px; padding: 20px; display: inline-block; text-align: left; margin: 20px auto; }
.grupo { margin-top: 24px; background: #FFF8E1; border-left: 5px solid #FFC107; padding: 14px 20px; border-radius: 0 8px 8px 0; }
.grupo h3 { margin: 0 0 8px; color: #F57F17; }
h2 { color: #1565C0; border-bottom: 2px solid #BBDEFB; padding-bottom: 4px; margin-top: 28px; }
h3 { color: #0D47A1; }
.ok   { background: #E8F5E9; border-left: 5px solid #43A047; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 14px 0; }
.warn { background: #FFF3E0; border-left: 5px solid #FB8C00; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 14px 0; }
img  { max-width: 88%; display: block; margin: 14px auto; border-radius: 6px; box-shadow: 0 2px 8px rgba(0,0,0,.15); }
.wide { max-width: 100%; }
.tbl { width: 100%; border-collapse: collapse; margin: 16px 0; font-size: 10.5pt; }
.tbl th { background: #1565C0; color: white; padding: 9px 14px; text-align: left; }
.tbl td { padding: 8px 14px; border-bottom: 1px solid #BBDEFB; }
.tbl tr:nth-child(even) td { background: #E3F2FD; }
.concl { background: #1565C0; color: white; padding: 18px 22px; border-radius: 8px; margin-top: 28px; }
.concl h2 { color: white; border: none; }
.footer { text-align: center; font-size: 9pt; color: #9E9E9E; margin-top: 36px; border-top: 1px solid #E0E0E0; padding-top: 10px; }
code { background: #F5F5F5; padding: 1px 5px; border-radius: 3px; font-family: monospace; font-size: 10pt; }
'''

HTML_DOC = f'''<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8"><style>{CSS}</style></head><body>

<div class="portada">
  <h1>Tema 1: Vale la pena agregar una tercera capa oculta?</h1>
  <h2>Comparacion de arquitecturas MLP para prediccion de abandono de clientes</h2>
  <div class="meta">
    <p><strong>Curso:</strong> Tecnicas Estadisticas Avanzadas para Mineria de Datos - ESTP1030</p>
    <p><strong>Maestria:</strong> Maestria en Estadistica Aplicada - ESPOL</p>
    <p><strong>Profesor:</strong> Dr. Andres Abad Robalino</p>
  </div>
  <div class="grupo">
    <h3>Integrantes del Grupo</h3>
    <p>VITERI RAMBAY IRWIN ALBERTO</p>
    <p>VICTORES SOLEDISPA LOURDES VANESSA</p>
    <p>GONZALEZ VILORIA GILBERTO ENRIQUE</p>
  </div>
</div>

<h2>1. Contexto</h2>
<p>Se compara el modelo base <code>(64, 32)</code> contra la arquitectura <code>(128, 64, 32)</code>. Metrica guia: <strong>F1-Score (clase Abandona)</strong>.</p>

<h2>2. Dataset: Telco Customer Churn</h2>
<p>7 043 clientes con variables demograficas, de servicios y de facturacion.</p>
<div class="warn"><strong>Desbalance:</strong> 73% No abandona | 27% Abandona.</div>
<img src="data:image/png;base64,{img_b64(F["dist"])}" style="max-width:52%;">

<h2>3. Preprocesamiento</h2>
<p>11 filas eliminadas (TotalCharges vacio). One-hot encoding: 20 a <strong>30 features</strong>. Division 80/20 estratificada. StandardScaler solo en train.</p>

<h2>4. Modelo Base (64, 32)</h2>
<img src="data:image/png;base64,{img_b64(F["cm_base"])}" style="max-width:52%;">
<div class="warn">Recall Abandona = 0.48 | <strong>F1 = 0.51</strong> (linea de referencia)</div>

<h2>5. Modelo Tema 1 (128, 64, 32)</h2>
<img src="data:image/png;base64,{img_b64(F["cm_t1"])}" style="max-width:52%;">
<div class="ok">Recall Abandona sube a 0.57 | <strong>F1 = 0.53</strong> (+4.68%)</div>

<h2>6. Comparacion de Resultados</h2>
{tabla_html}
<img src="data:image/png;base64,{img_b64(F["cm_cmp"])}" class="wide">
<img src="data:image/png;base64,{img_b64(F["roc"])}">
<img src="data:image/png;base64,{img_b64(F["f1"])}" style="max-width:52%;">

<h2>7. Analisis</h2>
<p>F1 sube de 0.5100 a 0.5338 (+4.68%). Recall +9pp. Accuracy baja levemente por mas falsos positivos, intercambio aceptable. AUC-ROC baja marginalmente de 0.7677 a 0.7590.</p>
<p>La capa de 128 neuronas aprende combinaciones que el modelo de dos capas no puede formar directamente. La mejora es modesta porque el churn depende de pocas variables clave ya capturadas por el modelo base.</p>

<div class="concl">
  <h2>8. Conclusion</h2>
  <p>La arquitectura <strong>(128, 64, 32) es la mejor opcion</strong> (Recall +9pp, F1 +4.68%).   El siguiente paso debe ser tratar el desbalance con <code>class_weight='balanced'</code>.</p>
</div>

<div class="footer">
  ESPOL | ESTP1030 | Prof. Dr. Andres Abad Robalino<br>
  Viteri Rambay I.A. | Victores Soledispa L.V. | Gonzalez Viloria G.E.
</div>
</body></html>'''

PDF_PATH = r'G:\Evaluacion\reporte_tema1_MLP.pdf'
WP_HTML(string=HTML_DOC).write_pdf(PDF_PATH)
print('PDF generado:')
print(' ', PDF_PATH)


PDF generado:
  G:\Evaluacion\reporte_tema1_MLP.pdf
